In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Data transformation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)

train_loader = DataLoader(dataset=train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=64, shuffle=False)
# Define CNN model
class CNN_CIFAR(nn.Module):
    def __init__(self):  # ✅ Fix: should be __init__, not _init_
        super(CNN_CIFAR, self).__init__()
        self.cnn1 = nn.Conv2d(3, 6, 5)
        self.cnn2 = nn.Conv2d(6, 16, 5)   
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.cnn1(x)
        x = torch.relu(x)
        x = torch.max_pool2d(x, 2)

        x = self.cnn2(x)
        x = torch.relu(x)
        x = torch.max_pool2d(x, 2)

        x = x.flatten(1)

        x = self.fc1(x)
        x = torch.relu(x)

        x = self.fc2(x)
        x = torch.relu(x)

        x = self.fc3(x)
        return x

# Initialize model, loss, and optimizer
model = CNN_CIFAR()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
print("")

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        images, labels = data

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(train_loader):.3f}")

# Evaluation
print("")

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print(f"Accuracy : {accuracy:.2f}%")



Epoch 1/10, Loss: 1.673
Epoch 2/10, Loss: 1.376
Epoch 3/10, Loss: 1.240
Epoch 4/10, Loss: 1.155
Epoch 5/10, Loss: 1.096
Epoch 6/10, Loss: 1.041
Epoch 7/10, Loss: 0.993
Epoch 8/10, Loss: 0.955
Epoch 9/10, Loss: 0.918
Epoch 10/10, Loss: 0.885

Accuracy : 64.22%
